# Validation with AHN5

At a chosen time `t` in the rosbag, this notebook:

1. Looks up the bike's GPS position at that time.
2. Estimates the bike's heading from a short window of GPS displacement.
3. Builds a sample strip along that heading from `X_BACK` (default −5 m, behind)
   to `X_FRONT` (default +25 m, in front), spaced every `DX` (default 0.5 m).
4. Queries the official Dutch AHN5 DSM **once** with a single bbox covering the
   whole strip and samples elevation at every point.
5. Returns a `profile_ahn5` dict with `x`/`z`/`slope_deg`/`kappa`

# TODO
Delete v7 strip code

In [ ]:
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory
import matplotlib.pyplot as plt
import numpy as np
from pyproj import Transformer
import requests
import rasterio
import io
import math
import scipy.interpolate as sc
# ============================================================
# SETTINGS — edit these before running
# ============================================================

GPS_FILE_PATH      = r"D:\Data_gathered\2026_05_01\Rosbag\14_30_00\rosbag\rosbag_0.mcap"
GPS_LAT_LON_TOPIC  = "/navsat_topic"           # set to None to list topics

TIME               = 1777638690.249665836

# Strip geometry  
X_BACK             = -5.0                      # m behind the bike
X_FRONT            = 20.0                      # m in front
DX                 = 0.5                       # interval between samples, this is the resolution of the AHN5 DSM heightmap

# Expected height profile, if it is not in this interval it will be seen as invaled and ignored
MIN_HEIGHT         = -50.0                      # m, expected minimum height of the terrain
MAX_HEIGHT         =  100.0                      # m, expected maximum height of the terrain
# Heading estimation
HEADING_WINDOW_S   = 1.0                       # ± seconds for GPS displacement
# Spline smoothing factor (higher = smoother, lower = closer fit to data)
SMOOTHENING_FACTOR = 0.005

## 1. Read GPS from the bag

In [ ]:
gps_coords = {"t": [], "lat": [], "lon": []}

print("Reading GPS bag:", GPS_FILE_PATH)

with open(GPS_FILE_PATH, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])

    if GPS_LAT_LON_TOPIC is None:
        print("\nGPS_LAT_LON_TOPIC is None. Available topics:")
        summary = reader.get_summary()
        for ch in summary.channels.values():
            n = summary.statistics.channel_message_counts.get(ch.id, "?")
            schema = summary.schemas.get(ch.schema_id, None)
            schema_name = schema.name if schema else "unknown"
            print(f"  {ch.topic}  —  {n} messages  [{schema_name}]")
        raise SystemExit("Set GPS_LAT_LON_TOPIC and re-run this cell.")

    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=[GPS_LAT_LON_TOPIC]):
        gps_coords["t"].append(message.publish_time / 1e9)
        gps_coords["lat"].append(ros_msg.latitude)
        gps_coords["lon"].append(ros_msg.longitude)

times = np.array(gps_coords["t"]) # Time is in UNIX format
lats  = np.array(gps_coords["lat"])
lons  = np.array(gps_coords["lon"])
if len(times) == 0:
    raise RuntimeError(f"No messages on topic '{GPS_LAT_LON_TOPIC}'. "
                       "Set GPS_LAT_LON_TOPIC = None to list available topics.")

print(f"  GPS messages   : {len(times)}")
print(f"  Bag time range : {times[0]:.2f}  →  {times[-1]:.2f}")
print(f"  Duration of route : {times[-1] - times[0]:.1f} s "
      f"({(times[-1] - times[0])/60:.1f} min)")


## 2. Convert to Dutch RD New (EPSG:28992)

AHN5 uses the dutch local coordinate system while the gps uses the global system.\
Here the gps coordinates get transformed into the local system.

In [ ]:
transformer = Transformer.from_crs("EPSG:4326", "EPSG:28992", always_xy=True)
rd_xs, rd_ys = transformer.transform(lons, lats)
rd_xs = np.array(rd_xs)
rd_ys = np.array(rd_ys)

print(f"  RD X range : {rd_xs.min():.1f}  →  {rd_xs.max():.1f}")
print(f"  RD Y range : {rd_ys.min():.1f}  →  {rd_ys.max():.1f}")

## 3. Pick the moment in time and estimate heading

Calculates the direction the bike was going over 1s, then it samples a point every DX for X_FRONT and X_BACK. This means it only looks forward.\
Direction comes from the bike's GPS displacement over `±HEADING_WINDOW_S` seconds.



In [ ]:
# pick the reference message 
ref_idx = np.argmin(np.abs(times - TIME)) #argmin returns the index of the minimum value in the array, which corresponds to the closest timestamp to TIME
dt = abs(times[ref_idx] - TIME)
print(f"Using TIME = {TIME}, closest GPS message Δt = {dt:.3f}s "
        f"(index {ref_idx})")

ref_t    = float(times[ref_idx])
ref_lat  = float(lats[ref_idx])
ref_lon  = float(lons[ref_idx])
ref_rd_x = float(rd_xs[ref_idx])
ref_rd_y = float(rd_ys[ref_idx])

# heading from GPS displacement over a short window
mask = (times >= ref_t - HEADING_WINDOW_S) & (times <= ref_t + HEADING_WINDOW_S)
win_x = rd_xs[mask]
win_y = rd_ys[mask]
if len(win_x) < 3: # Means the bike has barely moved or something went wrong with the GPS
    raise RuntimeError(
        f"Only {len(win_x)} GPS samples in ±{HEADING_WINDOW_S}s window. "
        "Increase HEADING_WINDOW_S.")

hd_x = win_x[-1] - win_x[0]
hd_y = win_y[-1] - win_y[0]
hd_norm = math.hypot(hd_x, hd_y) # hypot is the length of the vector from the origin to (hd_x, hd_y)

if hd_norm < 0.5:
    raise RuntimeError(
        f"Bike moved only {hd_norm*100:.1f} cm in window — "
        "stationary, heading undefined. Pick a different TIME.")

heading_x   = hd_x / hd_norm
heading_y   = hd_y / hd_norm
heading_deg = math.degrees(math.atan2(hd_y, hd_x))

print(f"  Reference position : ({ref_lat:.6f}, {ref_lon:.6f})")
print(f"  RD coordinates     : ({ref_rd_x:.2f}, {ref_rd_y:.2f})")
print(f"  Heading/direction  : {heading_deg:.1f}°  "
      f"(displacement {hd_norm:.2f} m over ±{HEADING_WINDOW_S}s)")

# sample points along the forward axis
x_samples = np.arange(X_BACK, X_FRONT + DX, DX)
sample_xs_rd = ref_rd_x + x_samples * heading_x
sample_ys_rd = ref_rd_y + x_samples * heading_y
print(f"  Strip from x={X_BACK}m to x={X_FRONT}m, {len(x_samples)} samples")


## 4. Query AHN5 DSM (one API call)

The whole strip fits in a small bounding box, so we issue **one** WCS request
covering it and sample every point from the returned GeoTIFF in memory.

In [ ]:
margin = 1.0    # m of padding around the strip
xmin = sample_xs_rd.min() - margin
xmax = sample_xs_rd.max() + margin
ymin = sample_ys_rd.min() - margin
ymax = sample_ys_rd.max() + margin

# AHN5 resolution = 0.5 m
res = 0.5
gw  = int(math.ceil((xmax - xmin) / res)) + 1
gh  = int(math.ceil((ymax - ymin) / res)) + 1
print(f"Querying AHN5: bbox ({xmin:.1f}, {ymin:.1f}) → ({xmax:.1f}, {ymax:.1f})")
print(f"  Grid size : {gw} × {gh}  ({gw*gh} pixels)")

AHN5_LAYERS = {
    "DSM": {
        "wcs_url":  "https://api.ellipsis-drive.com/v3/ogc/wcs/a4a8a27b-e36e-4dd5-a75b-f7b6c18d33ec",
        "coverage": "fc9d369f-94ca-4373-8281-a6854edb67c9",
    },
    "DTM": {
        "wcs_url":  "https://api.ellipsis-drive.com/v3/ogc/wcs/51e11c02-1065-463d-a0b0-263d80293b16",
        "coverage": "41eaf86f-fa84-48f7-a949-355044ed0e4e",
    },
}

def query_ahn5_layer(wcs_url, coverage, label):
    params = {
        "service":  "WCS",
        "version":  "1.0.0",
        "request":  "GetCoverage",
        "coverage": coverage,
        "crs":      "EPSG:28992",
        "bbox":     f"{xmin},{ymin},{xmax},{ymax}",
        "width":    gw,
        "height":   gh,
        "format":   "GEOTIFF",
    }
    resp = requests.get(wcs_url, params=params, timeout=30)
    resp.raise_for_status()
    print(f"  {label} returned {len(resp.content)} bytes")

    z = np.full(len(x_samples), np.nan)
    with rasterio.open(io.BytesIO(resp.content)) as src:
        no_data = src.nodata
        for i, vals in enumerate(src.sample(zip(sample_xs_rd, sample_ys_rd))):
            v = float(vals[0])
            if no_data is not None and v == no_data:
                continue
            if not np.isfinite(v):
                continue
            if MIN_HEIGHT < v < MAX_HEIGHT:
                z[i] = v
    return z

z_samples = {}
valid     = {}
for name, layer in AHN5_LAYERS.items():
    z_samples[name] = query_ahn5_layer(layer["wcs_url"], layer["coverage"], f"AHN5 {name}")
    valid[name]     = ~np.isnan(z_samples[name])
    n_valid = valid[name].sum()
    print(f"  {name} valid : {n_valid} / {len(z_samples[name])}", end="")
    if valid[name].any():
        print(f"  |  NAP range: {np.nanmin(z_samples[name]):.2f} → {np.nanmax(z_samples[name]):.2f} m")
    else:
        print()


## 5. Build `profile_ahn5` dict

The dict mirrors the v7 profile schema (`x`, `z`, `slope_deg`, `kappa`) so it
can be overlaid directly. The `z` field is **shifted so z(0) = 0** at the bike's
position — same convention as the v7 LiDAR-frame profile.
The original NAP elevations are kept as `z_absolute` for reference.

In [ ]:
def build_profile(name, z_raw):
    v = valid[name]
    if not v.all():
        z_filled = np.interp(x_samples, x_samples[v], z_raw[v])
    else:
        z_filled = z_raw.copy()

    ref_z_idx = int(np.argmin(np.abs(x_samples)))
    ref_z_nap = float(z_filled[ref_z_idx])
    z_relative = z_filled - ref_z_nap

    spline = sc.make_splrep(x_samples, z_relative, s=SMOOTHENING_FACTOR)
    dzdx   = spline.derivative(nu=1)(x_samples)
    d2zdx2 = spline.derivative(nu=2)(x_samples)
    slope_deg = np.degrees(np.arctan(dzdx))
    kappa     = abs(d2zdx2) / (1.0 + dzdx ** 2) ** 1.5

    return {
        "x":           x_samples,
        "z":           z_relative,
        "z_absolute":  z_filled,
        "slope_deg":   slope_deg,
        "kappa":       kappa,
        "r":           1.0 / kappa,
        "spline":      spline,
        "method":      f"AHN5 {name}",
        "ref_time":    ref_t,
        "ref_lat":     ref_lat,
        "ref_lon":     ref_lon,
        "ref_rd_x":    ref_rd_x,
        "ref_rd_y":    ref_rd_y,
        "ref_z_nap":   ref_z_nap,
        "heading_deg": heading_deg,
        "bin_size":    DX,
    }

profiles = {name: build_profile(name, z_samples[name]) for name in AHN5_LAYERS}

for name, p in profiles.items():
    print(f"  {name}: ref NAP = {p['ref_z_nap']:.2f} m | "
          f"z = {p['z'].min():+.3f} → {p['z'].max():+.3f} m | "
          f"slope = {p['slope_deg'].min():+.1f} → {p['slope_deg'].max():+.1f} deg")


## 6. Visualise the strip and the AHN5 profile

In [ ]:
LAYER_STYLE = {
    "DSM": {"color": "tab:green",  "raw_color": "darkgreen"},
    "DTM": {"color": "tab:orange", "raw_color": "darkorange"},
}

fig = plt.figure(figsize=(20, 10))
gs  = fig.add_gridspec(3, 2)

# --- top-left: full GPS track + strip ---
ax_map = fig.add_subplot(gs[0, 0])
ax_map.plot(rd_xs, rd_ys, "-", color="lightgray", lw=1, label="full GPS track")
ax_map.plot(sample_xs_rd, sample_ys_rd, "o-", color="red", ms=2, label="AHN5 strip")
ax_map.plot(ref_rd_x, ref_rd_y, "s", color="blue", ms=8, label="bike position")
ax_map.set_title("Top-down view (bag-wide)")
ax_map.set_xlabel("RD X [m]")
ax_map.set_ylabel("RD Y [m]")
ax_map.set_aspect("equal", adjustable="box")
ax_map.legend(loc="best", fontsize=9)
ax_map.grid(True, alpha=0.3)

# --- top-right: zoomed view of the strip ---
ax_zoom = fig.add_subplot(gs[0, 1])
ax_zoom.plot(sample_xs_rd, sample_ys_rd, "o-", color="red", ms=4, label="sample points")
ax_zoom.plot(ref_rd_x, ref_rd_y, "s", color="blue", ms=10, label="bike position")
ax_zoom.annotate("", xy=(ref_rd_x + 4*heading_x, ref_rd_y + 4*heading_y),
                 xytext=(ref_rd_x, ref_rd_y),
                 arrowprops=dict(arrowstyle="->", color="blue", lw=2))
ax_zoom.text(ref_rd_x + 4.5*heading_x, ref_rd_y + 4.5*heading_y,
             f"{heading_deg:.0f}°", color="blue")
ax_zoom.set_title("Sample strip (zoomed)")
ax_zoom.set_xlabel("RD X [m]")
ax_zoom.set_ylabel("RD Y [m]")
ax_zoom.set_aspect("equal", adjustable="box")
ax_zoom.legend(loc="best", fontsize=9)
ax_zoom.grid(True, alpha=0.3)

# --- 2nd row: elevation profiles ---
ax_p = fig.add_subplot(gs[1, :])
x_smooth = np.linspace(x_samples.min(), x_samples.max(), 1000)

for name, p in profiles.items():
    style = LAYER_STYLE[name]
    ref_z = p["ref_z_nap"]
    ax_p.plot(x_samples, p["z"], "-", color=style["color"], lw=2, label=f"AHN5 {name} (smoothed)")
    ax_p.plot(x_samples[valid[name]], (z_samples[name] - ref_z)[valid[name]],
              "o", color=style["raw_color"], ms=4, label=f"AHN5 {name} raw")
    ax_p.plot(x_smooth, p["spline"](x_smooth), "--", color=style["color"], lw=1, alpha=0.5)

ax_p.axvline(0, color="blue", lw=0.7, ls="--", label="bike at x=0")
ax_p.axhline(0, color="black", lw=0.5)
ax_p.set_title(f"AHN5 ground profile  —  t = {ref_t:.2f}  (frame {ref_idx})")
ax_p.set_xlabel("X (forward) [m]")
ax_p.set_ylabel("Elevation relative to bike [m]")
ax_p.legend(loc="best")
ax_p.grid(True, alpha=0.3)

# --- 3rd row: radius of curvature ---
ax_c = fig.add_subplot(gs[2, :])

for name, p in profiles.items():
    style = LAYER_STYLE[name]
    ax_c.plot(x_samples, p["r"], "-", color=style["color"], lw=2, label=f"AHN5 {name}")

ax_c.axvline(0, color="blue", lw=0.7, ls="--", label="bike at x=0")
ax_c.axhline(0, color="black", lw=0.5)
ax_c.set_title(f"AHN5 radius of curvature  —  t = {ref_t:.2f}  (frame {ref_idx})")
ax_c.set_xlabel("X (forward) [m]")
ax_c.set_ylabel("Radius [m]")
ax_c.legend(loc="best")
ax_c.grid(True, alpha=0.3)

plt.suptitle("AHN5 validation strip", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## SAVE output

In [ ]:
# ============================================================
# Save results to external drive
# ============================================================
import os

ROSBAG_FILE_PATH = GPS_FILE_PATH
USING_MCAP = True

if USING_MCAP:
    gps_parts = ROSBAG_FILE_PATH.replace("/", os.sep).split(os.sep)
    date_folder = gps_parts[-5]
    time_folder = gps_parts[-3]

SAVE_DIR = os.path.join(r"D:\Validation_results", date_folder, time_folder)
os.makedirs(SAVE_DIR, exist_ok=True)

def save_profile(p):
    t      = int(p["ref_time"])
    method = p["method"].replace(" ", "_")
    fpath  = os.path.join(SAVE_DIR, f"{method}_{t}.npz")
    np.savez_compressed(
        fpath,
        x          = p["x"],
        y          = None,
        z          = p["z"],
        s          = p["x"],
        t          = np.array([p["ref_time"]]),
        method     = np.array([p["method"]]),
        kappa      = p["kappa"],
        slope_deg  = p["slope_deg"],
        z_absolute = p["z_absolute"],
        heading_deg = np.array([p["heading_deg"]]),
        ref_lat    = np.array([p["ref_lat"]]),
        ref_lon    = np.array([p["ref_lon"]]),
        bin_size   = np.array([p["bin_size"]]),
    )
    print(f"Saved {p['method']} → {fpath}")
    return fpath

for name, p in profiles.items():
    save_profile(p)
